# OpenStudio Refrigeration System Modeling and JSON Generator 
<span style="font-size:100%; color:gray;">
SPDX-FileCopyrightText: 2025-present Oak Ridge National Laboratory, managed by UT-Battelle

SPDX-License-Identifier: BSD-3-Clause
</span>
# Step-by-Step Guide

<span style="font-size:18px"> This notebook provides a structured workflow to generate and export OpenStudio-compatible JSON files for supermarket refrigeration systems. 



🔑 **Key modules:**
- Compressor and performance curve generation
- Condenser and fan curve logic
- Rack assignment based on thermal loads
- Case and Walk-in object creation
- Full refrigeration system assembly and export

</span>

## Practical Applications

This framework can be applied to multiple real-world use cases:

- **Practical Refrigeration Design**  
  Engineers and energy consultants can use the generated components to design and evaluate supermarket refrigeration systems under different templates (old, new, advanced).

- **OpenStudio Energy Modeling**  
  The JSON objects generated here are fully compatible with OpenStudio's v0.2.1 schema, allowing integration into broader building energy simulation models. This is especially useful for load estimation, retrofit analysis, and performance benchmarking.

By automating object creation and tying it to real data and configurable templates, this tool serves as a bridge between design-level thinking and simulation-level precision.

### ⚙️ Mode Selection: How to Start
The modeling framework provides **two modes of operation** for flexibility and ease of use:

- **Automated Mode (Automated)**  
  This default mode automatically sets up a predefined SuperMarket configuration. It includes rack assignments, case/walk-in units, and refrigerant templates tailored for typical SuperMarket systems.  
  ✅ Ideal for quick scenario evaluations or template-based simulations.

- **User-Defined Mode (Manual Input)**  
  This mode allows the user to interactively choose specific refrigeration cases and walk-ins, define custom system configurations, and select desired templates.  
  ✅ Best for detailed, user-controlled modeling and custom design cases.
  
> These modes make the tool versatile for both **practical refrigeration system design** and **OpenStudio simulation workflows**, helping users make informed retrofit or maintenance decisions.

### 🧮 Rack Assignment Logic
Refrigeration cases and walk-ins are assigned to Medium or Low Temperature racks using a First Fit Decreasing (FFD) algorithm.

- Each MT/LT rack has a configurable capacity limit
- Loads are sorted and packed efficiently to minimize the number of racks.
- Large loads for same case type are split across racks
 
> Enables practical equipment sizing while supporting fast, scalable rack assignment.

### 🧰 Template Selection: System Type and Era

Once the mode is selected, the system **template** must be specified. This determines which performance curves, temperature assumptions, and equipment configurations are used for simulation and export.

Templates represent different eras of refrigeration system design:

- **`old`** : Representative systems based on equipment and configurations generally used **before 2010**  
- **`new`** : Representative systems based on equipment and configurations generally used **between 2010–2020**   
- **`advanced`** : Representative systems based on more recent equipment and system configurations **after 2020**  

> Your choice here influences curve data selection, compressor sizing, and condenser performance characteristics throughout the modeling workflow.

### ❄️ Temperature Levels: MT and LT Systems

In commercial refrigeration, systems are categorized by their suction temperature level, which determines their typical use cases and operating characteristics.

- **`MT`** (Medium Temperature) : Suction temp ≈ -6.7 °C (20 °F) ・ Condenser temp ≈ 48.9 °C (120 °F)

  ↳ Used for applications like dairy, produce, deli, and beverages
  ↳ Requires moderate compression ratios and operates with standard efficiency

- **`LT`** (Low Temperature) : Suction temp ≈ -31.7 °C (-25 °F) ・ Condenser temp ≈ 40.6 °C (105 °F)

  ↳ Used for frozen foods, ice cream, and long-term storage
  ↳ Operates with higher compression ratios and lower evaporating pressures

> These temperature values are used internally throughout the modeling process to select performance curves, size compressors, and evaluate condenser behavior.

### 🗺️ Zone Assignment for Refrigeration Units
Each refrigeration case and walk-in is assigned to a thermal zone in the OpenStudio model.

- **Cases** are placed in `"MainSales"` or a custom zone
- **Walk-ins** are placed in `"ActiveStorage"` or a user-defined zone

> Zones can be defined interactively (**user mode**) or selected automatically (**automated mode**).

## Step 1: Imports & Setup
Import required modules and set your database path.

In [1]:
import sys
sys.path.append(".")
from refrigeration.mode_selection import (
    automated_mode,
    user_mode,
    get_valid_template,
    select_test_mode
)
from refrigeration.rack_assignment import (
    distribute_units,
    assign_racks_to_cases_and_walkins,
    display_rack_capacity
)
from refrigeration.compressor import (
    summarize_compressor_assignment,
    prepare_and_store_compressor_objects,
    load_and_print_compressor_curves
)
from refrigeration.condenser import prepare_and_store_condenser_objects
from refrigeration.case_walkin_objects import (
    generate_case_objects_from_data,
    generate_walkin_objects_from_data,
    prepare_and_store_case_and_walkin_objects
)
from refrigeration.system_objects import (
    prepare_and_store_system_and_casewalkin_lists,
    generate_system_and_casewalkin_lists
)
from refrigeration.json_io import (
    export_existing_compressors_to_json,
    export_existing_condensers_to_json,
    export_cases_and_walkins_to_json,
    export_system_and_casewalkin_lists_to_json
)
from refrigeration.full_export import export_full_refrigeration_system_to_json
from refrigeration.utils import (
    get_building_name, set_mode,
    clean_name, generate_available_units_markdown
)

# set DB path 
db_path = "database/openstudio_refrigeration_system.db"

## Step 2: Select Mode (User / Automated)
Run either user_mode() or automated_mode() to proceed.

In this step, you’ll choose which refrigeration and freezer units to include and define the system template (Old, New, or Advanced).

- In **User mode**, you can manually select the units you want.

- In **Automated mode**, a default setup for the SuperMarket will be loaded automatically.

The selected template determines assumptions for performance curves and temperature settings, which will be used in later simulations and analysis.

<h2> Available Refrigeration Units</h2>
<h3> Available Cases by Template</h3>

<div style="display: flex; justify-content: space-between; gap: 4%;">

  <div style="width: 30%;">
    <h4>Old Template</h4>
    <ul>
      <li>LT Coffin - Ice Cream</li>
      <li>LT Reach-in - Frozen Food</li>
      <li>LT Reach-in - Ice Cream</li>
      <li>MT Island - Deli Produce</li>
      <li>MT Service - Meat Deli Bakery</li>
      <li>MT Vertical Open - All</li>
    </ul>
  </div>

  <div style="width: 30%;">
    <h4>New Template</h4>
    <ul>
      <li>LT Coffin - Ice Cream</li>
      <li>LT Reach-in - Frozen Food</li>
      <li>LT Reach-in - Ice Cream</li>
      <li>MT Island - Deli Produce</li>
      <li>MT Reach-in - Dairy Deli Beverage</li>
      <li>MT Service - Meat Deli Bakery</li>
      <li>MT Vertical Open - All</li>
    </ul>
  </div>

  <div style="width: 30%;">
    <h4>Advanced Template</h4>
    <ul>
      <li>LT Coffin - Ice Cream</li>
      <li>LT Reach-in - Frozen Food</li>
      <li>LT Reach-in - Ice Cream</li>
      <li>MT Island - Deli Produce</li>
      <li>MT Reach-in - Meat</li>
      <li>MT Reach-in - Others</li>
      <li>MT Service - Meat</li>
      <li>MT Service - Others</li>
      <li>MT Vertical Open - Beverage</li>
      <li>MT Vertical Open - Meat</li>
      <li>MT Vertical Open - Others</li>
    </ul>
  </div>

</div>

<h3> Available Walk-ins</h3>
<h4>For Old, New, and Advanced System Templates</h4>
* note: SF (Square Foot) indicates the physical floor area that the refrigeration unit is designed to serve
<div style="display: flex; justify-content: space-between; gap: 4%;">

  <div style="width: 48%;">
    <ul>
      <li>LT Walk-in Freezer - 80SF</li>
      <li>LT Walk-in Freezer - 120SF</li>
      <li>LT Walk-in Freezer - 240SF</li>
      <li>LT Walk-in Freezer - 360SF</li>
      <li>LT Walk-in Freezer - 480SF</li>
      <li>MT Walk-in Cooler - 64SF with no glass door</li>
      <li>MT Walk-in Cooler - 80SF with no glass door</li>    
      <li>MT Walk-in Cooler - 100SF with no glass door</li>
      <li>MT Walk-in Cooler - 120SF with no glass door</li>
      <li>MT Walk-in Cooler - 240SF with no glass door</li>
      <li>MT Walk-in Cooler - 360SF with no glass door</li>
      <li>MT Walk-in Cooler - 400SF with no glass door</li>
      <li>MT Walk-in Cooler - 480SF with no glass door</li>
      <li>MT Walk-in Cooler - 600SF with no glass door</li>
      <li>MT Walk-in Cooler - 660SF with no glass door</li>
    </ul>
  </div>

  <div style="width: 48%;">
    <ul>
      <li>MT Walk-in Cooler - 64SF with glass door</li>
      <li>MT Walk-in Cooler - 80SF with glass door</li> 
      <li>MT Walk-in Cooler - 100SF with glass door</li>
      <li>MT Walk-in Cooler - 120SF with glass door</li>
      <li>MT Walk-in Cooler - 240SF with glass door</li>
      <li>MT Walk-in Cooler - 360SF with glass door</li>
      <li>MT Walk-in Cooler - 400SF with glass door</li>
      <li>MT Walk-in Cooler - 480SF with glass door</li>
      <li>MT Walk-in Cooler - 600SF with glass door</li>
      <li>MT Walk-in Cooler - 660SF with glass door</li>
    </ul>
  </div>

</div>

### 📌 User Selection Mode – Input Format
Each **case** entry must follow this format:

$<$unit name$>$ 

$<$number of units$>$

Each **walk-in** entry only requires the name (unit count is fixed to 1):

$<$unit name$>$ 

#### 📌 **Examples**

**cases**

LT Coffin - Frozen Food 

2 

MT Vertical Open - Beverage

3

**walk-ins**

LT Walk-in Freezer - 240SF

MT Walk-in Cooler - 360SF with glass door

--> Please refer to `example_user_mode.ipynb` and `example_automated_mode.ipynb` for two example scenarios.

In [2]:
mode = select_test_mode()
set_mode(mode)

if mode == "user":
    selected_case_units, selected_walkin_units, selected_template, case_zone, walkin_zone = user_mode()
elif mode == "automated":
    selected_case_units, selected_walkin_units, selected_template, case_zone, walkin_zone = automated_mode(db_path)

# Selected Case Units
print("\nSelected Case Units:")
for unit in selected_case_units:
    print(f"\"osm name\": \"{unit.osm_name}\", \"case_name\": \"{unit.case_name}\", \"number_of_units\": {unit.number_of_units}")

# Selected Walk-in Units
print("\nSelected Walk-in Units:")
for unit in selected_walkin_units:
    print(f"\"osm name\": \"{unit.osm_name}\", \"walkin_name\": \"{unit.walkin_name}\", \"number_of_units\": {unit.number_of_units}")

Select test mode (user/automated):  automated


Choose building type:
1. SuperMarket
2. Convenience Store (Not Available yet)


Enter the number of your choice:  1


Chosen building type: SuperMarket


Choose template (old/new/advanced):  new



Selected Case Units:
"osm name": "SuperMarket new LT Coffin - Ice Cream - Ice Cream", "case_name": "new LT Coffin - Ice Cream", "number_of_units": 1.0
"osm name": "SuperMarket new LT Reach-In - Ice Cream - Ice Cream", "case_name": "new LT Reach-In - Ice Cream", "number_of_units": 14.0
"osm name": "SuperMarket new LT Reach-In - Frozen Food - Frozen Food", "case_name": "new LT Reach-In - Frozen Food", "number_of_units": 14.0
"osm name": "SuperMarket new MT Island - Deli Produce - Deli", "case_name": "new MT Island - Deli Produce", "number_of_units": 14.0
"osm name": "SuperMarket new MT Island - Deli Produce - Produce", "case_name": "new MT Island - Deli Produce", "number_of_units": 10.0
"osm name": "SuperMarket new MT Service - Meat Deli Bakery - Meat", "case_name": "new MT Service - Meat Deli Bakery", "number_of_units": 3.0
"osm name": "SuperMarket new MT Service - Meat Deli Bakery - Deli", "case_name": "new MT Service - Meat Deli Bakery", "number_of_units": 5.0
"osm name": "SuperMarke

## Step 3: Rack Assignment
This step determines how refrigeration loads are grouped into Medium Temperature (MT) and Low Temperature (LT) racks.

Refrigeration units are assigned to racks using a **First Fit Decreasing (FFD)** greedy algorithm:

Using the `assign_racks_to_cases_and_walkins()` function, it performs:

- ✅ Separates case and walk-in loads into MT and LT categories based on operation type  
- ✅ Automatically assigns and splits refrigeration loads into racks using a descending-capacity sort, creating new racks or appending [1], [2] when needed to stay within rack limits
- ✅ Stores rack assignment results for downstream compressor and condenser sizing

In [3]:
# Assign refrigeration units to racks and retrieve updated data
mt_racks, lt_racks, case_data, walkin_data = assign_racks_to_cases_and_walkins(
    db_path, selected_case_units, selected_walkin_units
)

# display capacity distribution for assigned racks
display_rack_capacity(mt_racks, {**case_data, **walkin_data}, rack_type="MT")
display_rack_capacity(lt_racks, {**case_data, **walkin_data}, rack_type="LT")


MT Racks:
Rack 1: Total Capacity = 49292.88 W
  - SuperMarket new MT Vertical Open - All - Dairy : 19951.68 W (7 units)
  - SuperMarket new MT Vertical Open - All - Prepared Foods : 19951.68 W (7 units)
  - SuperMarket new MT Island - Deli Produce - Deli : 9389.52 W (14 units)

Rack 2: Total Capacity = 49559.98 W
  - SuperMarket new MT Vertical Open - All - Meat : 17101.44 W (6 units)
  - SuperMarket new MT Vertical Open - All - Salad : 17101.44 W (6 units)
  - SuperMarket new MT Island - Deli Produce - Produce : 6706.80 W (10 units)
  - SuperMarket new MT Walk-in Cooler - 660SF with glass door - Dairy  : 6310.50 W (1 units)
  - SuperMarket new MT Walk-in Cooler - 120SF with no glass door - Deli  : 2339.80 W (1 units)

Rack 3: Total Capacity = 43737.94 W
  - SuperMarket new MT Walk-in Cooler - 600SF with no glass door - Meat Prep : 6162.00 W (1 units)
  - SuperMarket new MT Vertical Open - All - Beverage : 5700.48 W (2 units)
  - SuperMarket new MT Vertical Open - All - Floral : 5700.

## Step 4: Generate and Export Case and Walk-in objects
This step converts the **database-retrieved information** and **user-selected unit configurations** into OpenStudio-compatible `Refrigeration:Case` and `Refrigeration:WalkIn` JSON objects.

### Step 4.1: Generate Case and Walk-in JSON Objects

Using the `prepare_and_store_case_and_walkin_objects()` function, it:
- ✅ Maps each unit to its assigned thermal zone (e.g., user-defined zones such as `"Sales"`, `"Storage"`)
- ✅ Includes attributes such as cooling capacity, fan power, lighting, defrost schedule, etc.
- ✅ Generates OpenStudio-compatible object dictionaries used for JSON export

In [4]:
result = prepare_and_store_case_and_walkin_objects(case_data, walkin_data, selected_case_units, selected_walkin_units, case_zone, walkin_zone)
case_objects = result["case_objects"]
walkin_objects = result["walkin_objects"]

✅ Case and walk-in objects generated and stored


### Step 4.2: Export Case and Walk-in Objects in JSON Format

Using the `export_cases_and_walkins_to_json()` function, it:
- ✅ Adds user-defined or default zones (e.g., `"MainSales"`, `"ActiveStorage"`)
- ✅ Appends all `Refrigeration:Case` and `Refrigeration:WalkIn` objects
- ✅ Outputs a structured `.json` file for preview or integration with OpenStudio

In [5]:
export_cases_and_walkins_to_json(case_objects, walkin_objects, case_zone_name=case_zone, walkin_zone_name=walkin_zone, output_path="All_Cases_Walkins.json")

✅ Case + Walk-in JSON with zones saved to: All_Cases_Walkins.json

📦 Preview:
{
  "Version": "0.2.1",
  "Building": "SuperMarket",
  "objects": [
    {
      "type": "OS:ThermalZone",
      "name": "MainSales"
    },
    {
      "type": "OS:ThermalZone",
      "name": "ActiveStorage"
    },
    {
      "type": "OS:Refrigeration:Case",
      "name": "SuperMarket new LT Coffin - Ice Cream - Ice Cream",
      "ZoneName": "MainSales",
      "CaseName": "new LT Coffin - Ice Cream",
      "Template": "new",
      "OperationType": "LT",
      "RatedTotalCoolingCapacity": 240.4,
      "CaseLength": 2.0,
      "OperatingTemperature": -28.3,
      "EvaporatorTemperature": -31.1,
      "RatedLatentHeatRatio": 0.2,
      "RatedRuntimeFraction": 0.85,
      "LatentCaseCreditCurveType": "DewpointMethod",
      "LatentCaseCreditCurveName": "Coffin Latent Curve",
      "FanPowerPerUnitLength": 6.6,
      "LightingPowerPerUnitLength": 17.6,
      "CaseLightingScheduleName": null,
      "FractionofLight

## Step 5: Generate Compressor Objects
This step involves summarizing rack loads, retrieving compressor performance curves, creating compressor objects, and exporting them as OpenStudio-compatible JSON files.

### Step 5.1: Rack Assignment Summary
Using the `summarize_compressor_assignment()` function, it:
- ✅ Aggregates cooling loads across all assigned cases and walk-ins  
- ✅ Separates data into MT and LT groups  
- ✅ Outputs rack-wise load information required for compressor object generation

In [6]:
mt_info, lt_info = summarize_compressor_assignment(mt_racks, lt_racks, selected_template, db_path) 


🧊 MT Rack Compressor Assignment:
Rack 1: Load = 49292.88 W → Number of Compressors Needed = 3
Rack 2: Load = 49559.98 W → Number of Compressors Needed = 3
Rack 3: Load = 43737.94 W → Number of Compressors Needed = 3

❄️ LT Rack Compressor Assignment:
Rack 1: Load = 22925.30 W → Number of Compressors Needed = 3
Rack 2: Load = 24858.80 W → Number of Compressors Needed = 3

⚙️ Compressor Specs for the selected template 'new':
🧊 MT → Capacity: 38103.71 W, Power: 15447.52 W, COP: 2.47, EER: 8.42
❄️ LT → Capacity: 17182.20 W, Power: 9765.76 W, COP: 1.76, EER: 6.00


### Step 5.2: Compressor Curve Generation
Using the `load_and_print_compressor_curves()` function, it:
- ✅ Queries compressor curve data from the database  
- ✅ Extracts `OS:Curve:Bicubic` JSON objects for both power and capacity  
- ✅ Returns four curve objects ready for use in compressor creation

In [7]:
mt_power_curve, mt_capacity_curve, lt_power_curve, lt_capacity_curve = \
    load_and_print_compressor_curves(db_path, selected_template)

📈 MT Power Curve JSON:
{
    "type": "OS:Curve:Bicubic",
    "name": "new_Med_Temp_Comp_Pwr_Curve",
    "Coefficient1Constant": 4994.033647,
    "Coefficient2x": 170.249719,
    "Coefficient3x2": 2.846389112,
    "Coefficient4y": 356.7940341,
    "Coefficient5y2": -6.307470374,
    "Coefficient6xy": -1.409856005,
    "Coefficient7x3": 0.012160476,
    "Coefficient8y3": 0.076231729,
    "Coefficient9x2y": -0.057527471,
    "Coefficient10xy2": 0.009054446,
    "MinimumValueofx": -23.3,
    "MaximumValueofx": 7.2,
    "MinimumValueofy": 10.0,
    "MaximumValueofy": 60.0,
    "InputUnitTypeforX": "Temperature",
    "InputUnitTypeforY": "Temperature",
    "OutputUnitType": "Dimensionless"
}

📈 MT Capacity Curve JSON:
{
    "type": "OS:Curve:Bicubic",
    "name": "new_Med_Temp_Comp_Cap_Curve",
    "Coefficient1Constant": 61412.17466,
    "Coefficient2x": 1570.851906,
    "Coefficient3x2": 13.53344913,
    "Coefficient4y": -500.1952361,
    "Coefficient5y2": 8.16391243,
    "Coefficient6xy": 

### Step 5.3: Generate and Store Compressor Objects
Using the `prepare_and_store_compressor_objects()` function, it:
- ✅ Creates `OS:Refrigeration:Compressor` objects for MT and LT racks  
- ✅ Assigns suction temperatures and performance curves  
- ✅ Returns objects as structured Python dictionaries ready for export

In [8]:
result = prepare_and_store_compressor_objects(mt_info, lt_info, selected_template, db_path)
mt_compressors = result["mt_compressors"]
lt_compressors = result["lt_compressors"]
mt_power_curve = result["mt_power_curve"]
mt_capacity_curve = result["mt_capacity_curve"]
lt_power_curve = result["lt_power_curve"]
lt_capacity_curve = result["lt_capacity_curve"]

✅ Compressor objects and performance curves objects generated and stored in the result.


### Step 5.4: Preview and Export Compressor Objects in JSON Format
Using the `export_existing_compressors_to_json()` function, it:
- ✅ Includes all `OS:Refrigeration:Compressor` objects  
- ✅ Adds associated `OS:Curve:Bicubic` performance curves  
- ✅ Assigns user-defined or default thermal zones (e.g., `MainSales`, `Storage`)  
- ✅ Outputs a clean, OpenStudio-compatible `.json` file for integration or preview

In [9]:
export_existing_compressors_to_json(
    mt_compressors=mt_compressors,
    lt_compressors=lt_compressors,
    mt_power_curve=mt_power_curve,
    mt_capacity_curve=mt_capacity_curve,
    lt_power_curve=lt_power_curve,
    lt_capacity_curve=lt_capacity_curve,
    case_zone_name=case_zone, 
    walkin_zone_name=walkin_zone,
    output_path="All_Compressors.json"
)

✅ Compressor + Curve JSON with zones saved to: All_Compressors.json

📦 OpenStudio JSON Preview:

{
  "Version": "0.2.1",
  "Building": "SuperMarket",
  "objects": [
    {
      "type": "OS:ThermalZone",
      "name": "MainSales"
    },
    {
      "type": "OS:ThermalZone",
      "name": "ActiveStorage"
    },
    {
      "type": "OS:Curve:Bicubic",
      "name": "new_Med_Temp_Comp_Pwr_Curve",
      "Coefficient1Constant": 4994.033647,
      "Coefficient2x": 170.249719,
      "Coefficient3x2": 2.846389112,
      "Coefficient4y": 356.7940341,
      "Coefficient5y2": -6.307470374,
      "Coefficient6xy": -1.409856005,
      "Coefficient7x3": 0.012160476,
      "Coefficient8y3": 0.076231729,
      "Coefficient9x2y": -0.057527471,
      "Coefficient10xy2": 0.009054446,
      "MinimumValueofx": -23.3,
      "MaximumValueofx": 7.2,
      "MinimumValueofy": 10.0,
      "MaximumValueofy": 60.0,
      "InputUnitTypeforX": "Temperature",
      "InputUnitTypeforY": "Temperature",
      "OutputUnit

# Step 6: Generate Condenser Objects
This step handles the creation of air-cooled condenser objects and their associated fan power curves based on the rack load and system type (MT or LT).

### Step 6.1: Generate Condenser JSON Objects

Using the `prepare_and_store_condenser_objects()` function, it:
- ✅ Calculates condenser capacity and fan power based on MT/LT rack loads  
- ✅ Creates `OS:Refrigeration:Condenser:AirCooled` and `OS:Curve:Linear` objects  
- ✅ Returns Python dictionaries representing each condenser and its associated curve

In [10]:
result = prepare_and_store_condenser_objects(mt_info, lt_info, selected_template, db_path)
mt_condensers = result["mt_condensers"]
lt_condensers = result["lt_condensers"]
mt_curves = result["mt_curves"]
lt_curves = result["lt_curves"]

✅ Condenser and curve objects generated and stored in the result.


### Step 6.2: Export Condenser Objects in JSON Format

Using the `export_existing_condensers_to_json()` function, it:
- ✅ Includes all `OS:Refrigeration:Condenser:AirCooled` condenser objects  
- ✅ Adds associated `OS:Curve:Linear` fan power curves  
- ✅ Appends thermal zones (`MainSales` and `ActiveStorage`, or user-defined zones)  
- ✅ Outputs a structured, OpenStudio-compatible `.json` file for preview or integration

In [11]:
export_existing_condensers_to_json(
    mt_condensers=mt_condensers,
    lt_condensers=lt_condensers,
    mt_curves=mt_curves,
    lt_curves=lt_curves,
    case_zone_name=case_zone, 
    walkin_zone_name=walkin_zone,
    output_path="All_Condensers.json"
)

✅ Condensers + Curves with zones saved to: All_Condensers.json

📤 Condenser JSON Preview:
{
  "Version": "0.2.1",
  "Building": "SuperMarket",
  "objects": [
    {
      "type": "OS:ThermalZone",
      "name": "MainSales"
    },
    {
      "type": "OS:ThermalZone",
      "name": "ActiveStorage"
    },
    {
      "type": "OS:Refrigeration:Condenser:AirCooled",
      "name": "MT_Rack1_Condenser",
      "RatedEffectiveTotalHeatRejectionRate": 83131.88,
      "FanPower": 4361.12,
      "RatedSubcoolingTemperatureDifference": 5,
      "FanPowerCurve": "MT_Rack1_Condenser_FanCurve",
      "MinimumCondensingTemperature": 48.8889
    },
    {
      "type": "OS:Refrigeration:Condenser:AirCooled",
      "name": "MT_Rack2_Condenser",
      "RatedEffectiveTotalHeatRejectionRate": 83582.34,
      "FanPower": 4380.98,
      "RatedSubcoolingTemperatureDifference": 5,
      "FanPowerCurve": "MT_Rack2_Condenser_FanCurve",
      "MinimumCondensingTemperature": 48.8889
    },
    {
      "type": "OS:Re

## Step 7: Generate and Export Refrigeration Systems

This step creates `OS:Refrigeration:System` objects and their associated `OS:Refrigeration:CaseAndWalkInList` objects, linking refrigeration units to their assigned racks.

### Step 7.1: Generate Refrigeration System JSON Objects

Using the `prepare_and_store_system_and_casewalkin_lists()` function, it:

- ✅ Maps each case and walk-in unit to its assigned MT or LT rack  
- ✅ Assigns suction and condensing temperatures based on the selected system template  
- ✅ Creates `OS:Refrigeration:System` and `OS:Refrigeration:CaseAndWalkInList` objects  
- ✅ Returns Python dictionaries representing system configurations for JSON export

In [12]:
system_and_casewalkin_objects = prepare_and_store_system_and_casewalkin_lists(
    selected_case_units,
    selected_walkin_units,
    mt_racks,
    lt_racks,
    selected_template
)

✅ Refrigeration system +  case/walkin list objects generated and ready.


### Step 7.2: Export System and Case/Walk-in Lists in JSON Format

Using the `export_system_and_casewalkin_lists_to_json()` function, it:

- ✅ Includes all `OS:Refrigeration:System` and `OS:Refrigeration:CaseAndWalkInList` objects  
- ✅ Lists all associated case and walk-in names per system rack  
- ✅ Adds thermal zones (`MainSales` / `ActiveStorage` or user-defined)
- ✅ Outputs a structured, OpenStudio-compatible `.json` file for integration or preview

In [13]:
export_system_and_casewalkin_lists_to_json(
    system_and_casewalkin_objects,
    case_zone_name=case_zone,
    walkin_zone_name=walkin_zone,
    output_path="System_and_CaseWalkin_Lists.json"
)

✅ Refrigeration system + Case/Walk-in list saved to: System_and_CaseWalkin_Lists.json

📦 Preview:
{
  "Version": "0.2.1",
  "Building": "SuperMarket",
  "objects": [
    {
      "type": "OS:ThermalZone",
      "name": "MainSales"
    },
    {
      "type": "OS:ThermalZone",
      "name": "ActiveStorage"
    },
    {
      "type": "OS:Refrigeration:System",
      "name": "Supermarket Rack MT 1",
      "CompressorListName": "Compressor_List_MT_Rack1",
      "CondenserName": "MT_Rack1_Condenser",
      "CaseAndWalkInListName": "Supermarket Rack MT 1_CaseWalkinList",
      "RefrigerantType": "R404A",
      "SuctionTemperature": -6.6667,
      "MinimumCondensingTemperature": 48.8889,
      "EndUseSubcategory": "Refrigeration"
    },
    {
      "type": "OS:Refrigeration:CaseAndWalkInList",
      "name": "Supermarket Rack MT 1_CaseWalkinList",
      "CaseAndWalkInNames": [
        "SuperMarket new MT Vertical Open - All - Dairy",
        "SuperMarket new MT Vertical Open - All - Prepared Foo

# Step 8: Preivew and Export FULL Refrigeration system JSON Files
This final step consolidates all previously generated OpenStudio objects—including compressors, condensers, performance curves, refrigeration cases, walk-ins, and systems—into a single JSON file for full system integration or simulation.

Using the `export_full_refrigeration_system_to_json()` function, it:

- ✅ Combines all `OS:Refrigeration:Compressor`, `OS:Refrigeration:Condenser:AirCooled`, and related `OS:Curve` objects  
- ✅ Includes all `OS:Refrigeration:Case`, `OS:Refrigeration:WalkIn` objects with their thermal zone assignments  
- ✅ Adds `OS:Refrigeration:System` and `OS:Refrigeration:CaseAndWalkInList` objects for rack mapping 
- ✅ Outputs a complete, OpenStudio-compatible `.json` file for integration or preview


In [14]:
export_full_refrigeration_system_to_json(
    mt_compressors=mt_compressors,
    lt_compressors=lt_compressors,
    mt_power_curve=mt_power_curve,
    mt_capacity_curve=mt_capacity_curve,
    lt_power_curve=lt_power_curve,
    lt_capacity_curve=lt_capacity_curve,
    mt_condensers=mt_condensers,
    lt_condensers=lt_condensers,
    mt_curves=mt_curves,
    lt_curves=lt_curves,
    case_objects=case_objects,
    walkin_objects=walkin_objects,
    system_and_casewalkin_objects=system_and_casewalkin_objects,
    case_zone_name=case_zone,
    walkin_zone_name=walkin_zone,
    output_path="Full_Refrigeration_System.json"
)

print("🏁 Refrigeration JSON export complete.")
print(f"Building: {get_building_name()}")

✅ Full OpenStudio Refrigeration JSON saved to: Full_Refrigeration_System.json

📦 Preview:
{
  "Version": "0.2.1",
  "Building": "SuperMarket",
  "objects": [
    {
      "type": "OS:ThermalZone",
      "name": "MainSales"
    },
    {
      "type": "OS:ThermalZone",
      "name": "ActiveStorage"
    },
    {
      "type": "OS:Curve:Bicubic",
      "name": "new_Med_Temp_Comp_Pwr_Curve",
      "Coefficient1Constant": 4994.033647,
      "Coefficient2x": 170.249719,
      "Coefficient3x2": 2.846389112,
      "Coefficient4y": 356.7940341,
      "Coefficient5y2": -6.307470374,
      "Coefficient6xy": -1.409856005,
      "Coefficient7x3": 0.012160476,
      "Coefficient8y3": 0.076231729,
      "Coefficient9x2y": -0.057527471,
      "Coefficient10xy2": 0.009054446,
      "MinimumValueofx": -23.3,
      "MaximumValueofx": 7.2,
      "MinimumValueofy": 10.0,
      "MaximumValueofy": 60.0,
      "InputUnitTypeforX": "Temperature",
      "InputUnitTypeforY": "Temperature",
      "OutputUnitType": 